# Phase 3-4: ablation (base / +LLM / +stance / +LLM+stance) for every encoder

**Settings:** Accelerator **GPU T4 x2** and **Internet on**.
**Attach (Add Input → Your Work):** `phase3a-llm` (required), plus `phase1-baselines`, `phase1b-tuning` and `phase2-deberta`
so finished runs are reused.

Results are written to `/kaggle/working/outputs/ablation_weighted_f1.md`. If the session stops, attach this notebook's
last version as an input too and run it again. Finished runs are skipped.

In [ ]:
!nvidia-smi -L
import transformers, torch
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
# Clone (or update) the project code and the dataset.
import os, subprocess, glob
REPO = "akashnegetive/stance-aware-llm-explanation-augmented-hate-speech-detection-for-replies-in-conversation"
CODE_DIR = "/kaggle/working/code"
if os.path.isdir(CODE_DIR):
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", CODE_DIR], check=True)
%cd {CODE_DIR}
if not os.path.isdir("data/counter_context"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/xinchenyu/counter_context", "data/counter_context"], check=True)

In [ ]:
# One GPU per run keeps the batch size identical across all experiments.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
prev = sorted(set(glob.glob("/kaggle/input/*/outputs") + glob.glob("/kaggle/input/*/*/outputs")))
llm_files = sorted(glob.glob("/kaggle/input/*/llm/llm_outputs.jsonl") + glob.glob("/kaggle/input/*/*/llm/llm_outputs.jsonl"))
assert llm_files, "Attach the phase3a-llm notebook as an input first"
LLM_FILE = llm_files[0]
print("Reusing results from:", prev or "nothing yet")
print("LLM file:", LLM_FILE, "|", sum(1 for _ in open(LLM_FILE)), "lines (expect 4751)")

In [ ]:
# Smoke test (~3-5 min): 64 pairs, 1 epoch, all four variants. Catches problems before the long run.
!python ablation.py --models roberta-base=2e-5 --seeds 13 --limit 64 --llm_file {LLM_FILE} --output_dir /kaggle/working/smoke

In [ ]:
# Full ablation. Put the learning rate chosen in Phase 1b tuning after each model's "=".
# DeBERTa takes roughly twice as long as RoBERTa; remove it here to run RoBERTa only.
MODELS = "roberta-base=2e-5 microsoft/deberta-v3-base=2e-5"
!python ablation.py --models {MODELS} --select_metric weighted_f1 --stance_weight 0.5 --llm_file {LLM_FILE} --output_dir /kaggle/working/outputs --prev_results {" ".join(prev)}